# L6 — Multi-Agent Trace Walkthrough with Datadog APM

## What You'll Learn

- How a request flows through the multi-agent system end-to-end
- How to read a Datadog APM trace of the orchestrator's Strands agent loop (model calls, tool delegations, memory)
- What each span represents in the orchestration flow
- How to identify latency bottlenecks by span self-time

## Architecture Recap

The orchestrator is an AgentCore **Harness** running a managed **Strands agent loop**,
traced to Datadog via OTLP-direct. A single invocation looks like this in APM:

```
POST /invocations                          <- root span (the Harness invocation)
  invoke_agent (Strands Agents)            <- the managed agent loop
    execute_event_loop_cycle   (xN)        <- one reasoning iteration each
      chat -> ConverseStream (Bedrock)     <- the orchestrator's own LLM call
      execute_tool shell                   <- tool call / specialist delegation
      CreateEvent (AgentCore Memory)       <- short-term memory writes
```

**What you will NOT see in this trace:** the AgentCore Gateway and the tool Lambdas do
not emit spans into it -- trace context is not propagated past the managed Gateway, so a
specialist delegation shows only as the duration of an `execute_tool shell` span. The
specialist agents emit their **own** traces (query `service:anycompany-refund-agent`).

## Prerequisites

- L3 notebooks completed (agents deployed on Datadog sidecar with DD credentials in SSM)
- `DD_API_KEY` and `DD_APP_KEY` set as env vars (or stored in SSM)


## License Details

Refer to LICENSE file in the main folder for license details.

## Step 1: Setup

In [ ]:
import os, sys, time, uuid, json
from datetime import datetime, timedelta, timezone
import boto3
from botocore.config import Config
from datadog_api_client import Configuration, ApiClient
from datadog_api_client.v2.api.spans_api import SpansApi
from datadog_api_client.v2.model.spans_list_request import SpansListRequest
from datadog_api_client.v2.model.spans_list_request_data import SpansListRequestData
from datadog_api_client.v2.model.spans_list_request_attributes import SpansListRequestAttributes
from datadog_api_client.v2.model.spans_list_request_page import SpansListRequestPage
from datadog_api_client.v2.model.spans_sort import SpansSort

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'
DD_API_KEY = os.environ.get('DD_API_KEY', '')
DD_APP_KEY = os.environ.get('DD_APP_KEY', '')
DD_SITE = os.environ.get('DD_SITE', 'datadoghq.com')

if not DD_API_KEY:
    ssm = boto3.client('ssm', region_name=REGION)
    DD_API_KEY = ssm.get_parameter(Name=f'{SSM_PREFIX}/dd_api_key', WithDecryption=True)['Parameter']['Value']
    DD_APP_KEY = ssm.get_parameter(Name=f'{SSM_PREFIX}/dd_app_key', WithDecryption=True)['Parameter']['Value']

dd_config = Configuration()
dd_config.api_key['apiKeyAuth'] = DD_API_KEY
dd_config.api_key['appKeyAuth'] = DD_APP_KEY
dd_config.server_variables['site'] = DD_SITE
print('Datadog configured')


## Step 2: Invoke the Orchestrator

We send a refund question to the orchestrator (Harness). It runs its Strands agent
loop -- reasoning, calling the model, and delegating to specialist agent(s) via tool
calls -- which produces the rich orchestrator trace we inspect below.

In [ ]:
ssm = boto3.client('ssm', region_name=REGION)
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore = boto3.client('bedrock-agentcore', region_name=REGION, config=Config(read_timeout=300))

def get_ssm(n):
    return ssm.get_parameter(Name=n, WithDecryption=True)['Parameter']['Value']

cfg = {k: get_ssm(f'{SSM_PREFIX}/{k}') for k in ['harness_arn','cognito_client_id','user_password']}

cognito.initiate_auth(ClientId=cfg['cognito_client_id'], AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME':'gold_customer','PASSWORD':cfg['user_password']})
print('Authenticated')

SESSION_ID = str(uuid.uuid4()).upper()
ACTOR_ID = 'CUST-789'
# Refund question -> the orchestrator delegates to specialist agent(s) via tool calls.
TEST_PROMPT = 'Which of my orders are eligible for a refund?'

print(f'Prompt: {TEST_PROMPT}')
print(f'Session: {SESSION_ID}')

t0 = time.time()
resp = agentcore.invoke_harness(
    harnessArn=cfg['harness_arn'], runtimeSessionId=SESSION_ID, actorId=ACTOR_ID,
    messages=[{'role':'user','content':[{'text':f'[Authenticated customer: {ACTOR_ID} | Tier: gold]\n\n{TEST_PROMPT}'}]}])

text = ''
for ev in resp['stream']:
    if 'contentBlockDelta' in ev:
        text += ev['contentBlockDelta'].get('delta',{}).get('text','')

print(f'\nResponse ({time.time()-t0:.1f}s):')
print(text[:500])
print('\nWaiting 60s for trace propagation...')
time.sleep(60)


## Step 3: Retrieve the Trace from Datadog APM

In [ ]:
now = datetime.now(timezone.utc)

# --- Step 3a: FIND the orchestrator-rooted trace ---
# Query ONLY the orchestrator service (and drop /ping health checks). This reliably
# lands the top-level invocation trace (POST /invocations -> the Strands agent loop),
# instead of stray single-span fragments that "most spans" selection can pick.
FINDER_QUERY = 'service:anycompany-orchestrator -resource_name:"GET /ping"'

def _search_spans(query, minutes=15, limit=500):
    with ApiClient(dd_config) as client:
        body = SpansListRequest(data=SpansListRequestData(type='search_request',
            attributes=SpansListRequestAttributes(
                filter={'from': (now - timedelta(minutes=minutes)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                        'to': now.strftime('%Y-%m-%dT%H:%M:%SZ'), 'query': query},
                sort=SpansSort.TIMESTAMP_DESCENDING,
                page=SpansListRequestPage(limit=limit))))
        return [s.get('attributes', {}) for s in SpansApi(client).list_spans(body=body).get('data', [])]

finder_spans = _search_spans(FINDER_QUERY)
traces = {}
for a in finder_spans:
    traces.setdefault(a.get('trace_id', ''), []).append(a)

# Quality floor: a real orchestration trace has the Strands loop or several spans;
# reject trivial fragments (e.g. a lone `fastapi.serialize_response` span).
def _is_real(spans):
    names = {s.get('operation_name', '') for s in spans} | {s.get('resource_name', '') for s in spans}
    return bool(names & {'invoke_agent', 'execute_event_loop_cycle'}) or len(spans) >= 3

def _root_service(spans):
    ids = {s.get('span_id', '') for s in spans}
    for s in spans:
        if s.get('parent_id') in ('', '0', None) or s.get('parent_id') not in ids:
            return s.get('service')
    return None

candidates = {t: sp for t, sp in traces.items() if _is_real(sp)}
def _score(sp):
    return (_root_service(sp) == 'anycompany-orchestrator',
            len({s.get('service') for s in sp}), len(sp))
TRACE_ID = max(candidates, key=lambda t: _score(candidates[t])) if candidates else None

print(f'Finder query: {FINDER_QUERY}')
print(f'Found {len(finder_spans)} orchestrator spans in {len(traces)} traces '
      f'({len(candidates)} pass the quality floor)')

if not TRACE_ID:
    print()
    print('No representative orchestration trace in the window.')
    print('  -> Re-run Step 2 (invoke the Harness), wait ~30-60s, then re-run this cell.')
    print('     The full trace may have aged out of the 15-min window.')
else:
    # --- Step 3b: EXPAND by trace_id (no service filter) to pull EVERY span in the
    # trace across services (orchestrator + any specialist agent + ai_guard). ---
    trace_spans = [a for a in _search_spans(f'trace_id:{TRACE_ID}')
                   if a.get('resource_name') != 'GET /ping']
    if not trace_spans:                       # fall back to the finder spans we already have
        trace_spans = traces[TRACE_ID]
    traces[TRACE_ID] = trace_spans            # Step 4 reads traces[TRACE_ID]
    svcs = sorted({s.get('service', '?') for s in trace_spans})
    print(f'Selected trace: {TRACE_ID}')
    print(f'  spans (all services): {len(trace_spans)}')
    print(f'  services in trace:    {", ".join(svcs)}')

    # Flag an errored orchestration so a red demo trace is not confusing.
    if any(s.get('status') == 'error' for s in trace_spans):
        print()
        print('  NOTE: this trace ended in an ERROR (see the ERR status in Step 4) -')
        print('  a useful failure example. For a clean walkthrough, re-run Step 2 with a')
        print("  simple prompt (e.g. 'What is the status of order ORD-10001?'), wait ~60s,")
        print('  then re-run this cell.')


## Step 4: Display the Span Waterfall

Each span represents a step in the multi-agent flow. The indentation shows parent-child relationships.

In [ ]:
if not TRACE_ID:
    print('No trace. Re-run Step 3.')
else:
    # The v2 Spans API returns start_timestamp/end_timestamp (datetimes), not a
    # 'duration' field — compute duration from the two timestamps.
    def _ms(s):
        st, en = s.get('start_timestamp'), s.get('end_timestamp')
        try:
            return round((en - st).total_seconds() * 1000, 1) if (st and en) else 0.0
        except Exception:
            return 0.0
    tspans = sorted(traces[TRACE_ID],
                    key=lambda s: s.get('start_timestamp').timestamp() if s.get('start_timestamp') else 0.0)
    span_map = {s.get('span_id',''): s for s in tspans}
    children = {}
    roots = []
    for s in tspans:
        pid = s.get('parent_id','')
        if pid and pid in span_map:
            children.setdefault(pid,[]).append(s.get('span_id',''))
        else:
            roots.append(s.get('span_id',''))

    display = []
    def walk(sid, depth=0):
        s = span_map.get(sid, {})
        display.append({'name':s.get('resource_name',s.get('operation_name','?')),'service':s.get('service','?'),
                        'depth':depth,'ms':_ms(s),'status':s.get('status','ok')})
        for c in children.get(sid,[]):
            walk(c, depth+1)
    for r in roots:
        walk(r)
    if not display:
        display = [{'name':s.get('resource_name','?'),'service':s.get('service','?'),'depth':0,
                    'ms':_ms(s),'status':s.get('status','ok')} for s in tspans]

    print(f'Trace: {TRACE_ID}')
    print('='*90)
    print(f'{"Span":<42} {"Service":<20} {"Duration":>9} {"Status":>6}')
    print('-'*90)
    for i in display:
        indent = '  '*i['depth']
        pfx = '|-- ' if i['depth']>0 else ''
        n = f"{indent}{pfx}{i['name']}"[:41]
        st = 'ERR' if i['status']=='error' else 'ok'
        print(f"{n:<42} {i['service']:<20} {i['ms']:>7.1f}ms {st:>6}")
    print('='*90)


## Step 5: Understanding the Trace

This trace is found via the **orchestrator** service and then expanded by `trace_id`,
so the waterfall centers on the Harness's own managed **Strands agent loop** (exported
to Datadog via OTLP-direct). Here's what each span type means:

| Span | What it is |
|------|-----------|
| `POST /invocations` | The Harness receiving your `invoke_harness` call (root span) |
| `invoke_agent Strands Agents` | The Harness's **managed Strands agent loop**. It is AWS-run, so you cannot attach code or hooks here — this is exactly why L4 AI Guard runs on the specialist agents, not on the orchestrator. |
| `execute_event_loop_cycle` | One **reasoning iteration**: the orchestrator thinks, calls the model, maybe calls a tool, then repeats |
| `chat` -> `ConverseStream Bedrock Runtime` | The orchestrator's **own LLM call** (Claude deciding what to do) |
| `execute_tool shell` | The orchestrator **calling a tool / delegating to a specialist**. The long ones are specialist round-trips (the specialist's own work shows up here as duration). |
| `CreateEvent Bedrock AgentCore` | **AgentCore Memory** writing short-term conversation events (~100ms each) |

### What you will and won't see

- **Multiple `execute_event_loop_cycle` spans** mean the orchestrator iterated several
  times (think -> act -> observe) before answering.
- **This is an orchestrator-centric trace.** In practice the specialist agents emit
  their **own separate traces** -- the OTLP Harness pipeline and the ddtrace sidecar
  pipeline rarely share a `trace_id`, so "services in trace" usually shows only
  `anycompany-orchestrator`. To inspect a specialist's internals -- including its
  `ai_guard` and `acompletion` spans -- query it directly, e.g.
  `service:anycompany-refund-agent`. Step 3 still expands by `trace_id`, so on the
  rare occasion a specialist *does* join, it appears here automatically.
- **The Gateway and tool Lambdas do NOT appear as spans.** Trace context is not
  propagated past the AgentCore-managed Gateway, so a delegation shows only as the
  orchestrator's `execute_tool shell` duration -- not as gateway/Lambda/DynamoDB spans.
- **Two roots are normal.** The Harness (OTLP-direct) and any specialist agent (ddtrace
  sidecar) are separate telemetry pipelines; they share a `trace_id` but the
  cross-pipeline parent link may not resolve, so they can render as sibling roots.

### Where the time goes

The biggest costs are the `execute_tool shell` spans (specialist delegations) and the
`ConverseStream` LLM calls; memory `CreateEvent`s are cheap. Step 6 quantifies this with
**self-time** so parent/child durations are not double-counted.


## Step 6: Latency Breakdown by Service

In [ ]:
if TRACE_ID and traces.get(TRACE_ID):
    tspans = traces[TRACE_ID]
    span_map = {s.get('span_id', ''): s for s in tspans}

    def _ms(s):
        st, en = s.get('start_timestamp'), s.get('end_timestamp')
        try:
            return (en - st).total_seconds() * 1000 if (st and en) else 0.0
        except Exception:
            return 0.0

    # Self-time = a span's own duration MINUS the time covered by its direct children.
    # Summing raw span durations double-counts, because a parent span already includes
    # its children's time. Self-time attributes each millisecond to exactly one span.
    child_ms = {}
    for s in tspans:
        pid = s.get('parent_id', '')
        if pid in span_map:
            child_ms[pid] = child_ms.get(pid, 0.0) + _ms(s)

    rows = []
    for s in tspans:
        self_ms = max(_ms(s) - child_ms.get(s.get('span_id', ''), 0.0), 0.0)
        rows.append((s.get('resource_name', s.get('operation_name', '?')),
                     s.get('service', '?'), self_ms))

    total = sum(r[2] for r in rows)
    print('Self-time by span (top 12) - where wall-clock is actually spent:')
    print('-' * 72)
    for name, svc, ms in sorted(rows, key=lambda r: -r[2])[:12]:
        pct = f'{ms / total * 100:4.0f}%' if total else '   -'
        print(f'  {name[:34]:<34} {svc[:18]:<18} {ms:>8.0f}ms {pct}')
    print(f'  {"TOTAL (self-time)":<34} {"":<18} {total:>8.0f}ms')
    print()

    # Group self-time by operation kind for an honest bottleneck call-out.
    def _kind(name):
        n = name.lower()
        if 'converse' in n or 'chat' in n or 'completion' in n:
            return 'LLM inference'
        if 'tool' in n:
            return 'Tool / specialist delegation'
        if 'createevent' in n or 'memory' in n:
            return 'Memory writes'
        return 'Other / framework'

    by_kind = {}
    for name, svc, ms in rows:
        by_kind[_kind(name)] = by_kind.get(_kind(name), 0.0) + ms
    top_kind = max(by_kind, key=by_kind.get) if by_kind else None
    if top_kind and total:
        print(f'Bottleneck: {top_kind} ({by_kind[top_kind] / total * 100:.0f}% of self-time).')
        if top_kind == 'LLM inference':
            print('  -> Try a faster model, shorter prompts, or fewer reasoning iterations.')
        elif top_kind == 'Tool / specialist delegation':
            print('  -> Specialist round-trips dominate; parallelize or streamline agent calls.')
else:
    print('No trace selected - run Step 3 first.')


## Step 7: View in Datadog UI

For an interactive flame graph with full span details:

In [ ]:
if TRACE_ID:
    print(f'Open in browser:')
    print(f'  https://app.{DD_SITE}/apm/trace/{TRACE_ID}')
    print()
    print('In the Datadog UI you can:')
    print('  - Click any span to see metadata, tags, and related logs')
    print('  - Filter by error status to find failures')
    print('  - Compare traces side-by-side for performance regression')
    print('  - Set up trace-based monitors for alerting')


## Step 8: View in Datadog UI

The same trace is visible in the Datadog APM UI with a rich visual representation
showing the full span waterfall, service map, and timing breakdown.

Navigate to **APM > Traces** in your Datadog dashboard to see the end-to-end flow.

![Datadog Span Waterfall](datadog_span.png)


### End-to-End APM Trace in Datadog

The full trace shows Gateway → Lambda → DynamoDB with timing and status for each span.

![Datadog APM Trace](datadog_apm_trace.png)


## Summary

| What We Did | How |
|---|---|
| Invoked multi-agent flow | Prompt that hits Order + Refund agents |
| Retrieved trace | Datadog Spans API |
| Displayed waterfall | Parsed span hierarchy |
| Explained each service | Mapped spans to system components |
| Identified bottleneck | Latency breakdown by service |

### Next: `2_datadog_metrics_and_cost.ipynb`
